In [1]:
from pathlib import Path
import zipfile

import pandas as pd
import searoute as sr
from geopy.distance import great_circle
import coordinates as coord
import os
from core_modules import core_modules as core

In this code, I assemble the necessary data, clean it, conduct an exploratory data analysis, and then ...

# Data Collection

## Grid intensity

Grid intensity is important for manufacturing, but it's still relevant for supply chains. Source: [Ember Energy](https://ember-energy.org/latest-insights/global-electricity-review-2025/major-countries-and-regions/)



In [2]:
# gCO2/kWh - grid intensity is about CO2 released per unit of energy. Mostly about manufacturing, but still relevant
grid_intensity = {"china": 525, "mexico": 412, "s_korea": 390}


## Emission Factor
Ton-km emission factor is co2(tons)(tons)(tons)(tons)(tons)(tons)(tons) released per ton of commodity per kilometer transported. Relevant for transportation, depends on country's mix of transportation methods used. Because washing machines are transported mainly by trade vessels and trucks, these two are the main methods used in the calculation of the emission factor.

Lane-specific emission factors combine the IMO Fourth GHG Study global average with adjustments for typical vessel deployment on each lane (sourced from UNCTAD 2024 Chapter II) and feeder-megaship transshipment patterns documented in Notteboom & Rodrigue (2009).

Instead of country names, country codes were used as per [country.txt](https://www.census.gov/foreign-trade/schedules/c/country.txt) file on Census.gov.

Sources:
+ [US EPA SmartWay Carrier Emission Factors](epa.gov/smartway) - emission factor of Mexico-US trade routes
+ [New shipping routes highlight growing Asia-to-Mexico trade](https://www.freightwaves.com/news/new-shipping-routes-highlight-growing-asia-to-mexico-trade) - emission factor of Asia-Pacific trade routes
+ [Review of Maritime Transport 2024: Navigating Maritime Chokepoints](https://unctad.org/publication/review-maritime-transport-2024.) - GHG global average with country-specific adjustments for each trade route.
+ [Notteboom and Rodrigue](https://doi.org/10.1007/s10708-008-9210-4) - documents shipment patterns of feeder megaships

## Distance

This code calculates distances between ports for Asian origin countries (China, India, South Korea, Vietnam) using sea routes.

Because the PORTHS6MM dataset is encoded through fixed-width text, it is difficult to read and convert into dataframes. To solve this, I read it once, then saved the snapshot into Parquet. That way the parquet file is used for all subsequent runs.

This is the code that could be used to auto-import data. However, the problem with it is that there is poor connection between the API and the website.œ

In [3]:
# import requests
# from pathlib import Path

# OUT = Path("data/port_hs6")
# OUT.mkdir(parents=True, exist_ok=True)

# def url(year, month):
#     yy = str(year)[2:]
#     return (f"https://www.census.gov/trade/downloads/{year}"
#             f"/Port/im_hs6_m/PORTHS6MM{yy}{month:02d}.ZIP")

# # Your tariff treatment window
# for year in (2023, 2024, 2025):
#     for month in range(1, 13):
#         u = url(year, month)
#         out_path = OUT / f"PORTHS6MM{str(year)[2:]}{month:02d}.ZIP"
#         if out_path.exists():
#             continue
#         r = requests.get(u, timeout=30)
#         if r.status_code == 200:
#             out_path.write_bytes(r.content)
#             print(f"saved {out_path}")
#         else:
#             print(f"missing or not yet released: {u}")

# Data Cleaning
Cleaning is the longest and the most important step of any data cycle. After all, without good data there can be no good results. Because this projects uses data from a wide variety of sources, this means dealing with many different formats, which may complicate the cleaning process even further.

In [6]:
# this defines the columns in the PORTHS6MM dataframe, which are to be converted to numbers
numerical_cols_asian = ["gen_val_mo", "air_val_mo", "air_swt_mo", "ves_val_mo", "ves_swt_mo", "cnt_val_mo", "cnt_swt_mo"]


## Ocean-based Distance/Weight Data (Asian Routes, Pre-Tariff — Dec 2024)

In [7]:
_snapshot = Path('data/snapshots/PORTHS6MM2412.parquet')
if _snapshot.exists():
    df_ocean_routes_2412 = pd.read_parquet(_snapshot)
else:
    with zipfile.ZipFile('data/original/PORTHS6MM2412.ZIP') as zf:
        with zf.open(zf.namelist()[0]) as f:
            df_ocean_routes_2412 = pd.read_fwf(
                f,
                colspecs=coord.colspecs_sea,
                names=coord.names_sea,
                dtype={c: str for c in coord.str_cols_sea},
            )
    _snapshot.parent.mkdir(exist_ok=True)
    df_ocean_routes_2412.to_parquet(_snapshot, index=False)


In [8]:
df_ocean_routes_2412.info()

<class 'pandas.DataFrame'>
RangeIndex: 1212596 entries, 0 to 1212595
Data columns (total 20 columns):
 #   Column       Non-Null Count    Dtype
---  ------       --------------    -----
 0   commodity    1212596 non-null  str  
 1   cty_code     1212596 non-null  str  
 2   dist_unlade  1212596 non-null  str  
 3   port_unlade  1212596 non-null  str  
 4   year         1212596 non-null  str  
 5   month        1212596 non-null  str  
 6   gen_val_mo   1212596 non-null  int64
 7   air_val_mo   1212596 non-null  int64
 8   air_swt_mo   1212596 non-null  int64
 9   ves_val_mo   1212596 non-null  int64
 10  ves_swt_mo   1212596 non-null  int64
 11  cnt_val_mo   1212596 non-null  int64
 12  cnt_swt_mo   1212596 non-null  int64
 13  gen_val_yr   1212596 non-null  int64
 14  air_val_yr   1212596 non-null  int64
 15  air_swt_yr   1212596 non-null  int64
 16  ves_val_yr   1212596 non-null  int64
 17  ves_swt_yr   1212596 non-null  int64
 18  cnt_val_yr   1212596 non-null  int64
 19  cnt_swt_yr 

In [9]:
df_ocean_routes_2412["port_full"] = df_ocean_routes_2412["dist_unlade"] + df_ocean_routes_2412["port_unlade"]

df_asian_2412 = df_ocean_routes_2412[df_ocean_routes_2412["cty_code"].isin(coord.asian_origins)].T.drop_duplicates().T
for col in numerical_cols_asian:
    df_asian_2412[col] = pd.to_numeric(df_asian_2412[col])

df_asian_wash_2412 = df_asian_2412[df_asian_2412["commodity"].isin(["845011", "845020"])].copy()
df_asian_wash_2412 = df_asian_wash_2412[df_asian_wash_2412["port_full"].isin(coord.us_ports)]
df_asian_wash_2412 = df_asian_wash_2412[df_asian_wash_2412["ves_swt_mo"] > 0]
df_asian_wash_2412["distance"] = df_asian_wash_2412.apply(
    core.compute_distance, args=(coord.asian_origins, coord.us_ports, True, "cty_code", "port_full"), axis=1)
df_asian_wash_2412["co2"] = df_asian_wash_2412.apply(core.compute_co2, args=["ves_swt_mo"], axis=1)

df_asian_wash_2412.to_csv("data/intermediate/PORTHS6MM_asian_wash_2412.csv")
df_asian_wash_2412.info()

<class 'pandas.DataFrame'>
Index: 79 entries, 799729 to 800031
Data columns (total 23 columns):
 #   Column       Non-Null Count  Dtype  
---  ------       --------------  -----  
 0   commodity    79 non-null     object 
 1   cty_code     79 non-null     object 
 2   dist_unlade  79 non-null     object 
 3   port_unlade  79 non-null     object 
 4   year         79 non-null     object 
 5   month        79 non-null     object 
 6   gen_val_mo   79 non-null     int64  
 7   air_val_mo   79 non-null     int64  
 8   air_swt_mo   79 non-null     int64  
 9   ves_val_mo   79 non-null     int64  
 10  ves_swt_mo   79 non-null     int64  
 11  cnt_val_mo   79 non-null     int64  
 12  cnt_swt_mo   79 non-null     int64  
 13  gen_val_yr   79 non-null     object 
 14  air_val_yr   79 non-null     object 
 15  air_swt_yr   79 non-null     object 
 16  ves_val_yr   79 non-null     object 
 17  ves_swt_yr   79 non-null     object 
 18  cnt_val_yr   79 non-null     object 
 19  cnt_swt_yr   79 n

In [10]:
df_asian_wash_2412["port_full"].value_counts().sort_index(ascending=False)

port_full
5301    5
5201    3
4909    4
3002    5
3001    6
2904    3
2811    5
2709    6
2704    6
2002    1
1901    2
1803    4
1801    4
1703    6
1601    3
1401    4
1303    5
1003    6
0401    1
Name: count, dtype: int64

## Ocean-based Distance/Weight Data (Asian Routes, Post-Tariff — Dec 2025)

In [11]:
_snapshot = Path("data/snapshots/PORTHS6MM2512.parquet")
if _snapshot.exists():
    df_ocean_routes_2512 = pd.read_parquet(_snapshot)
else:
    with zipfile.ZipFile("data/original/PORTHS6MM2512.ZIP") as zf:
        with zf.open(zf.namelist()[0]) as f:
            df_ocean_routes_2512 = pd.read_fwf(
                f,
                colspecs=coord.colspecs_sea,
                names=coord.names_sea,
                dtype={c: str for c in coord.str_cols_sea},
            )
    _snapshot.parent.mkdir(exist_ok=True)
    df_ocean_routes_2512.to_parquet(_snapshot, index=False)

df_ocean_routes_2512["port_full"] = df_ocean_routes_2512["dist_unlade"] + df_ocean_routes_2512["port_unlade"]

df_asian_2512 = df_ocean_routes_2512[df_ocean_routes_2512["cty_code"].isin(coord.asian_origins)].T.drop_duplicates().T
for col in numerical_cols_asian:
    df_asian_2512[col] = pd.to_numeric(df_asian_2512[col])

df_asian_wash_2512 = df_asian_2512[df_asian_2512["commodity"].isin(["845011", "845020"])].copy()
df_asian_wash_2512 = df_asian_wash_2512[df_asian_wash_2512["port_full"].isin(coord.us_ports)]
df_asian_wash_2512 = df_asian_wash_2512[df_asian_wash_2512["ves_swt_mo"] > 0]
df_asian_wash_2512["distance"] = df_asian_wash_2512.apply(
    core.compute_distance, args=(coord.asian_origins, coord.us_ports, True, "cty_code", "port_full"), axis=1)
df_asian_wash_2512["co2"] = df_asian_wash_2512.apply(core.compute_co2, args=["ves_swt_mo"], axis=1)

df_asian_wash_2512.to_csv("data/intermediate/PORTHS6MM_asian_wash_2512.csv")
df_asian_wash_2512.info()

<class 'pandas.DataFrame'>
Index: 73 entries, 824153 to 824435
Data columns (total 23 columns):
 #   Column       Non-Null Count  Dtype  
---  ------       --------------  -----  
 0   commodity    73 non-null     object 
 1   cty_code     73 non-null     object 
 2   dist_unlade  73 non-null     object 
 3   port_unlade  73 non-null     object 
 4   year         73 non-null     object 
 5   month        73 non-null     object 
 6   gen_val_mo   73 non-null     int64  
 7   air_val_mo   73 non-null     int64  
 8   air_swt_mo   73 non-null     int64  
 9   ves_val_mo   73 non-null     int64  
 10  ves_swt_mo   73 non-null     int64  
 11  cnt_val_mo   73 non-null     int64  
 12  cnt_swt_mo   73 non-null     int64  
 13  gen_val_yr   73 non-null     object 
 14  air_val_yr   73 non-null     object 
 15  air_swt_yr   73 non-null     object 
 16  ves_val_yr   73 non-null     object 
 17  ves_swt_yr   73 non-null     object 
 18  cnt_val_yr   73 non-null     object 
 19  cnt_swt_yr   73 n

In [12]:
df_asian_wash_2512["port_full"].value_counts()

port_full
1003    6
1703    6
2704    6
1303    5
1801    5
2709    5
3002    5
1401    4
2811    4
1601    4
1803    4
5201    4
5301    4
3001    3
4909    3
1901    2
1001    1
2002    1
2904    1
Name: count, dtype: int64

## Ocean-based Distance/Weight Data (Asian Routes, Control — HS 8528 TVs, Dec 2024)

In [13]:
df_asian_tv_2412 = df_asian_2412[df_asian_2412["commodity"].str.startswith("8528")].copy()
df_asian_tv_2412 = df_asian_tv_2412[df_asian_tv_2412["port_full"].isin(coord.us_ports)]
df_asian_tv_2412 = df_asian_tv_2412[df_asian_tv_2412["ves_swt_mo"] > 0]
df_asian_tv_2412["distance"] = df_asian_tv_2412.apply(
    core.compute_distance, args=(coord.asian_origins, coord.us_ports, True, "cty_code", "port_full"), axis=1)
df_asian_tv_2412["co2"] = df_asian_tv_2412.apply(core.compute_co2, args=["ves_swt_mo"], axis=1)

df_asian_tv_2412.to_csv("data/intermediate/PORTHS6MM_asian_tv_2412.csv")
df_asian_tv_2412.info()

<class 'pandas.DataFrame'>
Index: 116 entries, 953212 to 955952
Data columns (total 23 columns):
 #   Column       Non-Null Count  Dtype  
---  ------       --------------  -----  
 0   commodity    116 non-null    object 
 1   cty_code     116 non-null    object 
 2   dist_unlade  116 non-null    object 
 3   port_unlade  116 non-null    object 
 4   year         116 non-null    object 
 5   month        116 non-null    object 
 6   gen_val_mo   116 non-null    int64  
 7   air_val_mo   116 non-null    int64  
 8   air_swt_mo   116 non-null    int64  
 9   ves_val_mo   116 non-null    int64  
 10  ves_swt_mo   116 non-null    int64  
 11  cnt_val_mo   116 non-null    int64  
 12  cnt_swt_mo   116 non-null    int64  
 13  gen_val_yr   116 non-null    object 
 14  air_val_yr   116 non-null    object 
 15  air_swt_yr   116 non-null    object 
 16  ves_val_yr   116 non-null    object 
 17  ves_swt_yr   116 non-null    object 
 18  cnt_val_yr   116 non-null    object 
 19  cnt_swt_yr   116

In [14]:
TvPortcodes2412 = df_asian_tv_2412["port_full"].value_counts().sort_index()
with pd.option_context('display.max_rows', None,
                       'display.max_columns', None,
                       'display.precision', 3,
                       ):
    print(TvPortcodes2412)

port_full
0401     2
1001     2
1003    11
1101     1
1303     2
1401     6
1501     1
1601     6
1703     9
1801     2
1803     1
1816     2
1901     2
2704    15
2709    15
2809     2
2811     7
3001     5
3002     8
3604     2
4909     3
5201     6
5203     1
5301     4
5310     1
Name: count, dtype: int64


## Ocean-based Distance/Weight Data (Asian Routes, Control — HS 8528 TVs, Dec 2025)

In [15]:
df_asian_tv_2512 = df_asian_2512[df_asian_2512["commodity"].str.startswith("8528")].copy()
df_asian_tv_2512 = df_asian_tv_2512[df_asian_tv_2512["port_full"].isin(coord.us_ports)]
df_asian_tv_2512 = df_asian_tv_2512[df_asian_tv_2512["ves_swt_mo"] > 0]
df_asian_tv_2512["distance"] = df_asian_tv_2512.apply(
    core.compute_distance, args=(coord.asian_origins, coord.us_ports, True, "cty_code", "port_full"), axis=1)
df_asian_tv_2512["co2"] = df_asian_tv_2512.apply(core.compute_co2, args=["ves_swt_mo"], axis=1)

df_asian_tv_2512.to_csv("data/intermediate/PORTHS6MM_asian_tv_2512.csv")
df_asian_tv_2512.info()

<class 'pandas.DataFrame'>
Index: 111 entries, 978494 to 981139
Data columns (total 23 columns):
 #   Column       Non-Null Count  Dtype  
---  ------       --------------  -----  
 0   commodity    111 non-null    object 
 1   cty_code     111 non-null    object 
 2   dist_unlade  111 non-null    object 
 3   port_unlade  111 non-null    object 
 4   year         111 non-null    object 
 5   month        111 non-null    object 
 6   gen_val_mo   111 non-null    int64  
 7   air_val_mo   111 non-null    int64  
 8   air_swt_mo   111 non-null    int64  
 9   ves_val_mo   111 non-null    int64  
 10  ves_swt_mo   111 non-null    int64  
 11  cnt_val_mo   111 non-null    int64  
 12  cnt_swt_mo   111 non-null    int64  
 13  gen_val_yr   111 non-null    object 
 14  air_val_yr   111 non-null    object 
 15  air_swt_yr   111 non-null    object 
 16  ves_val_yr   111 non-null    object 
 17  ves_swt_yr   111 non-null    object 
 18  cnt_val_yr   111 non-null    object 
 19  cnt_swt_yr   111

In [16]:
df_asian_tv_2512["port_full"].value_counts().sort_index()
TvPortcodes2512 = df_asian_tv_2512["port_full"].value_counts().sort_index()
with pd.option_context('display.max_rows', None,
                       'display.max_columns', None,
                       'display.precision', 3,
                       ):
    print(TvPortcodes2512)

port_full
0401     2
1001     4
1003     9
1102     1
1303     2
1401     5
1601     5
1703     8
1801     2
1816     1
1901     3
2002     1
2704    16
2709    13
2809     2
2811     9
2904     1
3001     6
3002     4
3604     1
4909     2
5201     5
5203     2
5301     5
5310     2
Name: count, dtype: int64


In [17]:
# import requests
# from pathlib import Path

# OUT = Path("data/port_hs6")
# OUT.mkdir(parents=True, exist_ok=True)

# def url(year, month):
#     yy = str(year)[2:]
#     return (f"https://www.census.gov/trade/downloads/{year}"
#             f"/Port/im_hs6_m/PORTHS6MM{yy}{month:02d}.ZIP")

# # Your tariff treatment window
# for year in (2023, 2024, 2025):
#     for month in range(1, 13):
#         u = url(year, month)
#         out_path = OUT / f"PORTHS6MM{str(year)[2:]}{month:02d}.ZIP"
#         if out_path.exists():
#             continue
#         r = requests.get(u, timeout=30)
#         if r.status_code == 200:
#             out_path.write_bytes(r.content)
#             print(f"saved {out_path}")
#         else:
#             print(f"missing or not yet released: {u}")

## Pre-Tariff Baseline — Dec 2022 (PORTHS6MM2212, parallel trends)

In [24]:
_snapshot = Path("data/snapshots/PORTHS6MM2212.parquet")
if _snapshot.exists():
    df_ocean_routes_2212 = pd.read_parquet(_snapshot)
else:
    with zipfile.ZipFile("data/original/PORTHS6MM2212.ZIP") as zf:
        with zf.open(zf.namelist()[0]) as f:
            df_ocean_routes_2212 = pd.read_fwf(
                f,
                colspecs=coord.colspecs_sea,
                names=coord.names_sea,
                dtype={c: str for c in coord.str_cols_sea},
            )
    _snapshot.parent.mkdir(exist_ok=True)
    df_ocean_routes_2212.to_parquet(_snapshot, index=False)
df_ocean_routes_2212["port_full"] = (
    df_ocean_routes_2212["dist_unlade"] + df_ocean_routes_2212["port_unlade"])

In [25]:
df_asian_2212 = df_ocean_routes_2212[
    df_ocean_routes_2212["cty_code"].isin(coord.asian_origins)
].T.drop_duplicates().T
for col in numerical_cols_asian:
    df_asian_2212[col] = pd.to_numeric(df_asian_2212[col])

df_asian_wash_2212 = df_asian_2212[df_asian_2212["commodity"].isin(["845011", "845020"])].copy()
df_asian_wash_2212 = df_asian_wash_2212[df_asian_wash_2212["port_full"].isin(coord.us_ports)]
df_asian_wash_2212 = df_asian_wash_2212[df_asian_wash_2212["ves_swt_mo"] > 0]
df_asian_wash_2212["distance"] = df_asian_wash_2212.apply(
    core.compute_distance, args=(coord.asian_origins, coord.us_ports, True, "cty_code", "port_full"), axis=1)
df_asian_wash_2212["co2"] = df_asian_wash_2212.apply(core.compute_co2, args=["ves_swt_mo"], axis=1)

df_asian_tv_2212 = df_asian_2212[df_asian_2212["commodity"].str.startswith("8528")].copy()
df_asian_tv_2212 = df_asian_tv_2212[df_asian_tv_2212["port_full"].isin(coord.us_ports)]
df_asian_tv_2212 = df_asian_tv_2212[df_asian_tv_2212["ves_swt_mo"] > 0]
df_asian_tv_2212["distance"] = df_asian_tv_2212.apply(
    core.compute_distance, args=(coord.asian_origins, coord.us_ports, True, "cty_code", "port_full"), axis=1)
df_asian_tv_2212["co2"] = df_asian_tv_2212.apply(core.compute_co2, args=["ves_swt_mo"], axis=1)

## Pre-Tariff Baseline — Dec 2023 (PORTHS6MM2312, parallel trends)

In [27]:
_snapshot = Path("data/snapshots/PORTHS6MM2312.parquet")
if _snapshot.exists():
    df_ocean_routes_2312 = pd.read_parquet(_snapshot)
else:
    with zipfile.ZipFile("data/original/PORTHS6MM2312.ZIP") as zf:
        with zf.open(zf.namelist()[0]) as f:
            df_ocean_routes_2312 = pd.read_fwf(
                f,
                colspecs=coord.colspecs_sea,
                names=coord.names_sea,
                dtype={c: str for c in coord.str_cols_sea},
            )
    _snapshot.parent.mkdir(exist_ok=True)
    df_ocean_routes_2312.to_parquet(_snapshot, index=False)
df_ocean_routes_2312["port_full"] = (
    df_ocean_routes_2312["dist_unlade"] + df_ocean_routes_2312["port_unlade"])

In [28]:
df_asian_2312 = df_ocean_routes_2312[
    df_ocean_routes_2312["cty_code"].isin(coord.asian_origins)
].T.drop_duplicates().T
for col in numerical_cols_asian:
    df_asian_2312[col] = pd.to_numeric(df_asian_2312[col])

df_asian_wash_2312 = df_asian_2312[df_asian_2312["commodity"].isin(["845011", "845020"])].copy()
df_asian_wash_2312 = df_asian_wash_2312[df_asian_wash_2312["port_full"].isin(coord.us_ports)]
df_asian_wash_2312 = df_asian_wash_2312[df_asian_wash_2312["ves_swt_mo"] > 0]
df_asian_wash_2312["distance"] = df_asian_wash_2312.apply(
    core.compute_distance, args=(coord.asian_origins, coord.us_ports, True, "cty_code", "port_full"), axis=1)
df_asian_wash_2312["co2"] = df_asian_wash_2312.apply(core.compute_co2, args=["ves_swt_mo"], axis=1)

df_asian_tv_2312 = df_asian_2312[df_asian_2312["commodity"].str.startswith("8528")].copy()
df_asian_tv_2312 = df_asian_tv_2312[df_asian_tv_2312["port_full"].isin(coord.us_ports)]
df_asian_tv_2312 = df_asian_tv_2312[df_asian_tv_2312["ves_swt_mo"] > 0]
df_asian_tv_2312["distance"] = df_asian_tv_2312.apply(
    core.compute_distance, args=(coord.asian_origins, coord.us_ports, True, "cty_code", "port_full"), axis=1)
df_asian_tv_2312["co2"] = df_asian_tv_2312.apply(core.compute_co2, args=["ves_swt_mo"], axis=1)

## Weight Data(USA Trade Online) - irrelevant?

Already well-organized, the weight data does not need much cleaning.

However, the previous datasets(US Census) contains everything necessary. Not just weight of imports, but also origin and destination info. Use this as a sanity check for weight of ocean-side imports.

In [30]:
washing_machine_df_v1 = washing_machine_df.drop("Unnamed: 5", axis=1)
washing_machine_df_v1["Air SWT (kg)"] = washing_machine_df_v1["Air SWT (kg)"].str.replace(',', '')
washing_machine_df_v1["Vessel SWT (kg)"] = washing_machine_df_v1["Vessel SWT (kg)"].str.replace(',', '')

washing_machine_df_v1["Air SWT (kg)"] = pd.to_numeric(washing_machine_df_v1["Air SWT (kg)"])
washing_machine_df_v1["Vessel SWT (kg)"] = pd.to_numeric(washing_machine_df_v1["Vessel SWT (kg)"])

washing_machine_df_v1 = washing_machine_df_v1.fillna(0)

In [31]:
washing_machine_df_v1.head()

,Commodity,Country,Time,Air SWT (kg),Vessel SWT (kg)
0,845011 Washing Mach Automatic W Dry Line Cap N...,China,January 2025,174.0,1777528.0
1,845011 Washing Mach Automatic W Dry Line Cap N...,China,February 2025,0.0,2459827.0
2,845011 Washing Mach Automatic W Dry Line Cap N...,China,March 2025,0.0,2268490.0
3,845011 Washing Mach Automatic W Dry Line Cap N...,China,April 2025,0.0,2512291.0
4,845011 Washing Mach Automatic W Dry Line Cap N...,China,May 2025,0.0,2541012.0


In [32]:
washing_machine_df_v1[washing_machine_df_v1["Country"]=="Mexico"].head()

,Commodity,Country,Time,Air SWT (kg),Vessel SWT (kg)
32,845011 Washing Mach Automatic W Dry Line Cap N...,Mexico,January 2025,0.0,55246.0
33,845011 Washing Mach Automatic W Dry Line Cap N...,Mexico,February 2025,0.0,74445.0
34,845011 Washing Mach Automatic W Dry Line Cap N...,Mexico,March 2025,0.0,239852.0
35,845011 Washing Mach Automatic W Dry Line Cap N...,Mexico,April 2025,0.0,175512.0
36,845011 Washing Mach Automatic W Dry Line Cap N...,Mexico,May 2025,0.0,199598.0


In [33]:
washing_machine_df_v1["Time"].value_counts()

Time
April 2025            9
June 2025             9
July 2025             9
October 2025          9
December 2025         9
2026 through March    9
January 2026          9
January 2025          8
February 2025         8
March 2025            8
May 2025              8
August 2025           8
September 2025        8
November 2025         8
February 2026         8
March 2026            8
Name: count, dtype: int64

In [34]:
washing_machine_df_v1["Air SWT (kg)"].mean()

np.float64(171.28148148148148)

# CO2 Calculations(transportation)

Now that we have the three necessary components - ton-km CO2 factor, weight of imports and distance of imports, then we can calculate the carbon footprint of transportation. This is only one step, as we also need to calculate CO2 of manufacturing(grid intensity x energy spent on manufacturing) in order to understand the full extent of the carbon footprint.

CO2, in this case, is measured in grams. For now, we do this only for 5 countries(China, Vietnam, South Korea, India, Mexico) and only for the month of January 2025, using the datasets taken from census.gov website.

In [35]:
# df_asian_routes_wash["co2"] = df_asian_routes_wash["vessel_swt_mo"]*df_asian_routes_wash["distance"]*ton_km[df_asian_routes_wash["cty_code"]]
df_asian_tv_2412["co2"] = df_asian_tv_2412.apply(core.compute_co2, args = ["ves_swt_mo"], axis=1)


In [36]:
df_asian_tv_2412.head()

,commodity,cty_code,dist_unlade,port_unlade,year,month,gen_val_mo,air_val_mo,air_swt_mo,ves_val_mo,...,gen_val_yr,air_val_yr,air_swt_yr,ves_val_yr,ves_swt_yr,cnt_val_yr,cnt_swt_yr,port_full,distance,co2
953212,852849,5700,27,09,2024,12,136068,0,0,136068,...,174835,0,0,174835,27408,174835,27408,2709,10674.961095,8.998138e+05
953794,852852,5520,17,03,2024,12,626316,0,0,626316,...,2608368,0,0,2608368,79266,2608368,79266,1703,20949.030244,3.620474e+06
953797,852852,5520,27,04,2024,12,35278490,0,0,35278490,...,425218271,0,0,425218271,23321912,422194514,23231547,2704,13525.167789,2.115113e+08
953798,852852,5520,27,09,2024,12,1250476,0,0,1250476,...,52803271,0,0,52803271,2479649,52803271,2479649,2709,13525.167789,3.925464e+06
953803,852852,5520,30,02,2024,12,1117640,0,0,1117640,...,3321417,0,0,3321417,114877,3321417,114877,3002,12411.976275,3.666411e+06


In [37]:
df_asian_tv_2412.info()

<class 'pandas.DataFrame'>
Index: 116 entries, 953212 to 955952
Data columns (total 23 columns):
 #   Column       Non-Null Count  Dtype  
---  ------       --------------  -----  
 0   commodity    116 non-null    object 
 1   cty_code     116 non-null    object 
 2   dist_unlade  116 non-null    object 
 3   port_unlade  116 non-null    object 
 4   year         116 non-null    object 
 5   month        116 non-null    object 
 6   gen_val_mo   116 non-null    int64  
 7   air_val_mo   116 non-null    int64  
 8   air_swt_mo   116 non-null    int64  
 9   ves_val_mo   116 non-null    int64  
 10  ves_swt_mo   116 non-null    int64  
 11  cnt_val_mo   116 non-null    int64  
 12  cnt_swt_mo   116 non-null    int64  
 13  gen_val_yr   116 non-null    object 
 14  air_val_yr   116 non-null    object 
 15  air_swt_yr   116 non-null    object 
 16  ves_val_yr   116 non-null    object 
 17  ves_swt_yr   116 non-null    object 
 18  cnt_val_yr   116 non-null    object 
 19  cnt_swt_yr   116

# Statistics

Now we use the difference-in-difference method to evaluate the extent, to which the tariffs have changed the CO2 footprint. We compare the differences in the control group and the treatment group. 

What will be the control group in our case? 

The control group will be "commodity", more specifically TV sets. TVs have mostly the same demand as washing machines, since both are bought together when people move to a new home. However, unlike the washing machines, TV sets don't fall under the Liberation Day tariffs and steel tariffs of 2025, which is due to goods with semiconductors being exempt from tariffs. These factors make TVs a good control group, since they're similar to washing machines in most key regards, except for the thing that we try to measure.

However, TV sets are lighter than washing machines, so CO2 change scales differently. One way to fix this is to normalize CO2 data by calculating CO2/kg of commodity, rather than total CO2. 


This code sets up a DiD model and runs an OLS regression on **Asian sea-route data only**. Each row is a unique combination of (origin country, US port of unlading, commodity, period). Each row carries total weight of products imported, total CO2 released, and weighted-average distance.

In [ ]:
def _tag(df, commodity, period, weight_col="ves_swt_mo"):
    out = df[["cty_code", weight_col, "co2", "distance", "port_full"]].copy()
    out = out.rename(columns={weight_col: "weight"})
    out["commodity"] = commodity
    out["period"] = period
    return out

df_sea_all = pd.concat([
    _tag(df_asian_wash_2412, "washing_machine", "2412"),
    _tag(df_asian_wash_2512, "washing_machine", "2512"),
    _tag(df_asian_tv_2412,   "tv",              "2412"),
    _tag(df_asian_tv_2512,   "tv",              "2512"),
    _tag(df_asian_wash_2212, "washing_machine", "2212"),
    _tag(df_asian_wash_2312, "washing_machine", "2312"),
    _tag(df_asian_tv_2212,   "tv",              "2212"),
    _tag(df_asian_tv_2312,   "tv",              "2312"),
], ignore_index=True)

df_sea_all["dist_x_weight"] = df_sea_all["distance"] * df_sea_all["weight"]

df_sea_agg = df_sea_all.groupby(["cty_code", "commodity", "period", "port_full"]).agg(
    total_weight=("weight", "sum"),
    total_co2=("co2", "sum"),
    _dist_x_wt=("dist_x_weight", "sum"),
).reset_index()

df_sea_agg["avg_distance"] = df_sea_agg["_dist_x_wt"] / df_sea_agg["total_weight"]
df_sea_agg = df_sea_agg.drop(columns="_dist_x_wt")
df_sea_agg

In [39]:
import statsmodels.api as sm

df_did = df_sea_agg[df_sea_agg["total_weight"] > 0].copy()
df_did["co2_intensity"] = df_did["total_co2"] / df_did["total_weight"]
df_did["treated"] = (df_did["commodity"] == "washing_machine").astype(int)
df_did["post"]    = (df_did["period"]    == "2512").astype(int)
df_did["s"] = df_did["cty_code"].map(coord.tariff_rate) * df_did["treated"]
df_did["treated_x_post"] = df_did["s"] * df_did["post"]

df_did[["cty_code", "port_full", "co2_intensity", "treated", "post", "treated_x_post"]].sort_values("co2_intensity", ascending=False).head(10)

,cty_code,port_full,co2_intensity,treated,post,treated_x_post
91,2010,3310,1201.052679,0,1,0.00
70,2010,3126,562.331561,0,0,0.00
16,2010,3126,562.331561,0,0,0.00
103,2010,4909,374.910450,1,0,0.00
97,2010,4909,374.910450,0,1,0.00
110,2010,4909,374.910450,1,0,0.00
124,2010,4909,374.910450,1,1,0.04
47,2010,4909,374.910450,0,0,0.00
118,2010,4909,374.910450,1,0,0.00
21,2010,4909,374.910450,0,0,0.00


In [40]:
df_sea_all.head()

,cty_code,weight,co2,distance,port_full,commodity,period,dist_x_weight
0,5520,223317.0,3.130660e+07,20027.006554,1003,washing_machine,2412,4.472371e+09
1,5520,2683.0,3.866830e+05,20589.054455,1303,washing_machine,2412,5.524043e+07
2,5520,4488.0,6.581347e+05,20949.030244,1703,washing_machine,2412,9.401925e+07
3,5520,13663.0,2.034897e+06,21276.406623,1801,washing_machine,2412,2.906995e+08
4,5520,89496.0,8.473139e+06,13525.167789,2704,washing_machine,2412,1.210448e+09


In [41]:
# CO2 intensity variation by origin country — a non-zero std means the DiD has signal to work with
df_did.groupby(["cty_code", "commodity"])["co2_intensity"].std().describe()

count      9.000000
mean      47.735903
std       55.534898
min       11.957109
25%       17.895028
50%       19.893426
75%       27.262073
max      152.854105
Name: co2_intensity, dtype: float64

A sanity check. If std = 0 within every (port_full, commodity) group, the DiD has no signal. Non-zero std here reflects cross-country variation in emission factors and route lengths across the four Asian origins.

In [43]:
sea_check = df_sea_agg.copy()
sea_check["co2_intensity"] = sea_check["total_co2"] / sea_check["total_weight"]

sea_check.groupby(["port_full", "commodity"])["co2_intensity"].std().describe()

count    7.300000e+01
mean     2.451903e+01
std      3.642518e+01
min      0.000000e+00
25%      3.552714e-15
50%      3.191601e+00
75%      3.402201e+01
max      1.619634e+02
Name: co2_intensity, dtype: float64

In [44]:
X = sm.add_constant(df_did[["treated", "post", "treated_x_post"]])
model = sm.OLS(df_did["co2_intensity"], X, missing="drop").fit(cov_type="HC3")
print(model.summary())

                            OLS Regression Results                            
Dep. Variable:          co2_intensity   R-squared:                       0.022
Model:                            OLS   Adj. R-squared:                  0.016
Method:                 Least Squares   F-statistic:                     5.470
Date:                Sun, 09 Aug 2026   Prob (F-statistic):            0.00105
Time:                        10:30:36   Log-Likelihood:                -2999.8
No. Observations:                 505   AIC:                             6008.
Df Residuals:                     501   BIC:                             6025.
Df Model:                           3                                         
Covariance Type:                  HC3                                         
                     coef    std err          z      P>|z|      [0.025      0.975]
----------------------------------------------------------------------------------
const            120.3470      6.209     19.

In [45]:
# Event-study parallel-trends test (reference period: 2412)
df_did["d2212"] = (df_did["period"] == "2212").astype(int)
df_did["d2312"] = (df_did["period"] == "2312").astype(int)

df_did["treat_x_d2212"] = df_did["s"] * df_did["d2212"]
df_did["treat_x_d2312"] = df_did["s"] * df_did["d2312"]

X_es = sm.add_constant(df_did[[
    "treated", "d2212", "d2312", "post",
    "treat_x_d2212", "treat_x_d2312", "treated_x_post"
]])
model_es = sm.OLS(df_did["co2_intensity"], X_es, missing="drop").fit(cov_type="HC3")
print(model_es.summary())

print("\nJoint pre-trend F-test (treat_x_d2212 = treat_x_d2312 = 0):")
print(model_es.f_test("treat_x_d2212 = 0, treat_x_d2312 = 0"))

                            OLS Regression Results                            
Dep. Variable:          co2_intensity   R-squared:                       0.023
Model:                            OLS   Adj. R-squared:                  0.010
Method:                 Least Squares   F-statistic:                     3.119
Date:                Sun, 09 Aug 2026   Prob (F-statistic):            0.00313
Time:                        10:30:36   Log-Likelihood:                -2999.5
No. Observations:                 505   AIC:                             6015.
Df Residuals:                     497   BIC:                             6049.
Df Model:                           7                                         
Covariance Type:                  HC3                                         
                     coef    std err          z      P>|z|      [0.025      0.975]
----------------------------------------------------------------------------------
const            118.7885     11.428     10.

Having run the OLS regression on the difference-in-differences model, we can assess whether the 2025 tariffs caused a statistically significant change in CO2 intensity for washing machines relative to TVs across the four Asian origin countries. The event-study cell below provides the parallel trends test.